# California County Spending Efficiency

## Karen Wu

In [1]:
import pandas as pd
import sqlite3

# Load your raw files
revenues = pd.read_csv('data/county_revenues.csv')
expenditures = pd.read_csv('data/county_expenditures.csv')

# Create a SQLite database file
conn = sqlite3.connect('data/ca_county_finances.db')

# Write each dataframe into the database as a table
revenues.to_sql('revenues', conn, if_exists='replace', index=False)
expenditures.to_sql('expenditures', conn, if_exists='replace', index=False)

conn.close()
print("Database created successfully")

DatabaseError: Execution failed on sql 'DROP TABLE "revenues"': database is locked

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/ca_county_finances.db')

final_query = """
WITH yoy_trends AS (
    SELECT 
        county,
        fiscal_year,
        net_total,
        population,
        net_total - LAG(net_total) OVER (PARTITION BY county ORDER BY fiscal_year) AS yoy_change
    FROM county_finances_combined
),
yoy_trends_per_capita AS (
    SELECT 
        county,
        fiscal_year,
        ROUND(CAST(yoy_change AS FLOAT) / population, 2) AS yoy_change_per_capita
    FROM yoy_trends
    WHERE yoy_change IS NOT NULL
),
avg_trend AS (
    SELECT 
        county,
        AVG(yoy_change_per_capita) AS avg_yearly_change_per_capita,
        SQRT(
            AVG(yoy_change_per_capita * yoy_change_per_capita) 
            - AVG(yoy_change_per_capita) * AVG(yoy_change_per_capita)
        ) AS volatility,
        COUNT(*) AS years_counted
    FROM yoy_trends_per_capita
    GROUP BY county
)
SELECT 
    county,
    ROUND(avg_yearly_change_per_capita, 2) AS avg_change,
    ROUND(volatility, 2) AS volatility,
    ROUND(avg_yearly_change_per_capita / volatility, 3) AS reliability_ratio,
    years_counted,
    RANK() OVER (ORDER BY avg_yearly_change_per_capita DESC) AS naive_rank,
    RANK() OVER (ORDER BY avg_yearly_change_per_capita / volatility DESC) AS reliability_rank
FROM avg_trend
ORDER BY reliability_rank;
"""

trend_results = pd.read_sql_query(final_query, conn)

# Also export the full year-by-year combined data for time-series visuals in Tableau
yearly_data = pd.read_sql_query("SELECT * FROM county_finances_combined ORDER BY county, fiscal_year", conn)

conn.close()

# Export both for Tableau
trend_results.to_csv('data/county_trend_reliability.csv', index=False)
yearly_data.to_csv('data/county_yearly_finances.csv', index=False)

print("Exported both files successfully")
print(f"Trend reliability rankings: {trend_results.shape[0]} counties")
print(f"Yearly finances detail: {yearly_data.shape[0]} rows")

Exported both files successfully
Trend reliability rankings: 57 counties
Yearly finances detail: 1254 rows
